<a target="_blank" href="https://colab.research.google.com/drive/1f19fAnhV55LZLuGMQCW1t9YZ8CZ_CElg?usp=sharing">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Fine-tune Gemma in Keras using LoRA

### Configure your API key

To use Gemma, you must provide your Kaggle username and a Kaggle API key.

To generate a Kaggle API key, go to the **Account** tab of your Kaggle user profile and select **Create New Token**. This triggers the download of a `kaggle.json` file containing your API credentials.

In Colab, select **Secrets** (🔑) in the left pane and add your Kaggle username and Kaggle API key. Store your username under the name `KAGGLE_USERNAME` and your API key under the name `KAGGLE_KEY`.

### Set environment variables

Set environment variables for `KAGGLE_USERNAME` and `KAGGLE_KEY`.

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### Install Keras packages

Install the Keras and KerasHub Python packages.

In [2]:
!pip install -q -U keras-hub
!pip install  -q -U keras

### Select a backend

Keras is a high-level, multi-framework deep learning API designed for simplicity and ease of use. Using Keras 3, you can run workflows on one of three backends: TensorFlow, JAX, or PyTorch. For this tutorial, configure the backend for JAX as it typically provides the better performance.

In [3]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

### Import packages

Import the Python packages needed for this tutorial, including Keras and KerasHub.

In [4]:
import keras
import keras_hub

## Load model

Keras provides implementations of Gemma and many other popular [model architectures](https://keras.io/keras_hub/api/models/). Use the `Gemma3CausalLM.from_preset()` method to configure an end-to-end Gemma model for causal language modeling. A causal language model predicts the next token based on previous tokens.

In [5]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

The `Gemma3CausalLM.from_preset()` method instantiates the model from a preset architecture and weights. In the code above, the string `"gemma#_xxxxxxx"` specifies a preset version and parameter size for Gemma. You can find the code strings for Gemma models in their **Model Variation** listings on [Kaggle](https://www.kaggle.com/models/keras/gemma3).

## Inference before fine tuning

Once you have downloaded and configured a Gemma model, you can query it with various prompts to see how it responds.

### Before Finetuning

The model clearly doesn't output a commit, rather it treats it as a Chat QA session

In [6]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="Fix self-attention bug",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=256))

Instruction:
Fix self-attention bug

Response:
```python
import torch
import torch.nn.functional as F

def fix_self_attention_bug(input_tensor, attention_mask):
    """
    Fixes the self-attention bug in a neural network.

    Args:
        input_tensor (torch.Tensor): The input tensor to be processed.
        attention_mask (torch.Tensor): The attention mask to be applied.

    Returns:
        torch.Tensor: The modified input tensor.
    """
    # Placeholder: Replace with your actual fix
    print("Self-attention bug fixed (placeholder).")
    return input_tensor
```

The provided code attempts to fix a self-attention bug by simply returning the input tensor without any modifications.
However, the actual self-attention bug is a subtle issue where the attention weights are being calculated incorrectly due to a lack of proper attention masking.  It's not a simple fix.
The code is meant to serve as a starting point for debugging and understanding the problem. 

A more robust solution 

## LoRA fine-tuning

This section shows you how to do fine-tuning using the Low Rank Adaptation (LoRA) tuning technique. This approach allows you to change the behavior of Gemma models using fewer compute resources.

### Format tuning data

Format the downloaded data for use with the Keras `fit()` method. The following code extracts a subset of the training examples to execute the notebook faster. Consider using more training data for higher quality fine-tuning.

In [7]:
!git clone https://github.com/neel04/pomni.git

fatal: destination path 'pomni' already exists and is not an empty directory.


In [8]:
import json

filename = "./pomni/data/500_documented_commits.json"
prompts = []
responses = []
line_count = 0

with open(filename) as file:
    data_list = json.load(file)

for examples in data_list:
    prompts.append(examples["text_input"])
    responses.append(examples["output"])
    line_count += 1

data = {
    "prompts": prompts,
    "responses": responses
}

### Configure LoRA tuning

Activate LoRA tuning using the Keras `model.backbone.enable_lora()` method, including a LoRA rank value. The *LoRA rank* determines the dimensionality of the trainable matrices that are added to the original weights of the LLM. It controls the expressiveness and precision of the fine-tuning adjustments. A higher rank means more detailed changes are possible, but also means more trainable parameters. A lower rank means less computational overhead, but potentially less precise adaptation.

This example uses a LoRA rank of 4. In practice, begin with a relatively small rank (such as 4, 8, 16). This setting is computationally efficient for experimentation. Train your model with this rank and evaluate the performance improvement on your task. Gradually increase the rank in subsequent trials and see if that further boosts performance.

In [9]:
gemma_lm.backbone.enable_lora(rank=16)

Check the model summary after setting the LoRA rank. Notice that enabling LoRA reduces the number of trainable parameters significantly compared to the total number of parameters in the model:

In [10]:
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,002,495,104 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,002,495,104 (3.73 GB)

 Trainable params: 2,609,152 (9.95 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

Configure the rest of the fine-tuning settings, including the preprocessor settings, optimizer, number of tuning epochs, and batch size:

In [11]:
gemma_lm.preprocessor.sequence_length = 384

optimizer = keras.optimizers.AdamW(
    learning_rate=6e-4,
    weight_decay=1e-3,
)
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

#### Mixed precision fine-tuning on NVIDIA GPUs

Full precision is recommended for fine-tuning. When fine-tuning on NVIDIA GPUs, you can use mixed precision (`keras.mixed_precision.set_global_policy('mixed_bfloat16')`) to speed up training with minimal effect on training quality.

In [12]:
keras.mixed_precision.set_global_policy('mixed_bfloat16')

### Run the fine-tune process

Run the fine-tuning process using the `fit()` method. This process can take several minutes depending on your compute resources, data size, and number of epochs:

In [13]:
history = gemma_lm.fit(data, epochs=8, batch_size=4)

Epoch 1/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - loss: 1.5305 - sparse_categorical_accuracy: 0.6167
Epoch 2/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 78s 543ms/step - loss: 1.1522 - sparse_categorical_accuracy: 0.6720
Epoch 3/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 543ms/step - loss: 1.0668 - sparse_categorical_accuracy: 0.6877
Epoch 4/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 543ms/step - loss: 0.9860 - sparse_categorical_accuracy: 0.7039
Epoch 5/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 543ms/step - loss: 0.9167 - sparse_categorical_accuracy: 0.7173
Epoch 6/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 544ms/step - loss: 0.8651 - sparse_categorical_accuracy: 0.7253
Epoch 7/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 543ms/step - loss: 0.8222 - sparse_categorical_accuracy: 0.7336
Epoch 8/8
95/95 ━━━━━━━━━━━━━━━━━━━━ 53s 543ms/step - loss: 0.7870 - sparse_categorical_accuracy: 0.7424


## Inference after fine-tuning

After fine-tuning, you should see changes in the responses when the tuned model is given the same prompt.

In [14]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="Fix self-attention bug",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=1)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=1024))

Instruction:
Fix self-attention bug

Response:
```diff
Commit: a94401e042214410e95000445212214e002040e9
Date: 2025-05-14T01:45:04Z
URL: https://github.com/jax-ml/jax/commit/a94401e042214410e95000445212214e002040e9
Files changed: 1
Additions: +2, Deletions: -2
diff --git a/jax/_src/pallas/pallas.py b/jax/_src/pallas/pallas.py
index 511440a2e2c4..0000e5043051 100644
--- a/jax/_src/pallas/pallas.py
+++ b/jax/_src/pallas/pallas.py
@@ -157,12 +157,12 @@ def process_query_helper(
     return jaxpr.new_jaxpr_call(
-        jaxpr.xsum_to_jnp(x, axis_range=(0, 1)),
-        name="identity_weight_before_eq",
-        suppress_errors=True,
-    )
   
   # TODO(yashkatariya): Implement a self-attention fix for this bug
-  pass
